# PushT Demo 

## Setup

In [16]:
import json
import subprocess
import time
from pathlib import Path

import gymnasium as gym
import gym_pusht
import numpy as np
import torch

WEEK = Path("/Users/snudurupati/Projects/physical-ai-data-architect/weeks/week-03")
MIG  = str(WEEK / "model" / "diffusion_pusht_migrated")
SIM  = str(WEEK / "data" / "pusht_sim")
EVAL_DIR = WEEK / "outputs" / "dp_official_eval" / "videos" / "pusht_0"

## Part 3: the environment

PushT (`gym_pusht`, Pymunk 2-D physics, not MuJoCo): a pusher drags a T-block to a goal pose.

- Action = absolute target pixel in [0, 512]². The pusher chases it with a PD controller (k_p=100, k_v=20); `step()` = 3 physics substeps.
- Observation here = 5 numbers: pusher (x, y), block (x, y), block angle.
- Goal is fixed at [256, 256, π/4]. Success = coverage > 0.95 of the block's **center of mass**, which sits ~45 px above the geometric center.
- Gravity = 0, damping = 0: the block floats where it starts; only contact moves it.
- Episode cap: 300 steps. `info["coverage"]` and `info["is_success"]` are free ground-truth labels from the physics engine.

In [17]:
env = gym.make("gym_pusht/PushT-v0", obs_type="state", render_mode="rgb_array")
obs, info = env.reset(seed=0)

In [18]:
print("action space low / high (absolute target pixels):")
print(env.action_space.low, env.action_space.high)

action space low / high (absolute target pixels):
[0. 0.] [512. 512.]


In [19]:
print("obs (pusher_x, pusher_y, block_x, block_y, block_angle):")
print(obs)
print("goal_pose:", info["goal_pose"])

obs (pusher_x, pusher_y, block_x, block_y, block_angle):
[390.         304.         241.54255641 268.51697354   3.3990369 ]
goal_pose: [256.         256.           0.78539816]


## Part 4: The baseline: A Random Policy

`run_policy` rolls any `make_action(obs, info)` for `n_episodes` episodes and reports successes, mean failure length, and mean best coverage.

- Action: a fresh uniform sample in [0, 512]² every frame. No state read, no memory.
- Expected output: `0/100 successes, mean failure length 300, mean max coverage ~0.11` — coverage starts near 0.1 and nothing ever reaches 0.95.
- The point of the baseline: every later number is measured against it.

In [6]:
def run_policy(make_action, n_episodes=100):
    successes, fail_lengths, max_covs = 0, [], []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        done, n, best = False, 0, 0.0
        while not done and n < 300:
            obs, r, term, trunc, info = env.step(make_action(obs, info))
            best = max(best, info["coverage"])
            done = term or trunc
            n += 1
        successes += int(term)
        if not term:
            fail_lengths.append(n)
        max_covs.append(best)
    print(f"{successes}/{n_episodes} successes, "
          f"mean failure length {np.mean(fail_lengths):.0f}, "
          f"mean max coverage {np.mean(max_covs):.3f}")

run_policy(lambda obs, info: env.action_space.sample().astype(np.float32))

0/100 successes, mean failure length 300, mean max coverage 0.115


## Part 5: the cowboy policy

The plan: stand on the far side of the block from the goal and shove it over.<br>
 Code it as: aim the pusher at `block + 40 px` along the block→goal direction.<br>

![The Plan](../assets/policy_v1.png)


In [ ]:
def cowboy_policy(margin):
    def policy(obs, info):
        goal = np.array([info["goal_pose"][0], info["goal_pose"][1]])
        block = np.array([obs[2], obs[3]])
        d = block - goal
        norm = np.linalg.norm(d)
        u = d / norm if norm > 0 else np.array([0.0, 0.0])
        return (block + u * margin).astype(np.float32)
    return policy

run_policy(cowboy_policy(margin=40))

In [ ]:
run_policy(cowboy_policy(margin=10))   # the "cowboy": spins the T

<video src="../assets/cowboy.mov" loop autoplay muted controls width="342" height="342" style="object-fit: cover;"></video><br>
<br>Why it fails — the physics has opinions:

- **margin 40:** the target lands on the block, so the pusher homes in, presses, and stops. Force through a PD controller with no momentum: it holds position, it does not sweep. Block never moves; coverage freezes at its spawn value (0.27).
- **margin 10:** the pusher now catches the T's thin arm. Force applied off the center of mass makes torque τ = r × F. With no gravity and no friction to pin the block, torque spins it instead of translating it. The T rotates in place forever.
- **What we assumed vs what is true:** we assumed pushing the middle moves the block to the goal; the block only translates when the force line passes through its center of mass, and coverage is measured *on the center of mass*, which is not the geometric center.
- Takeaway: a plausible diagram is not a control law. Contact geometry decides.

One margin-40 episode, printed every 50 steps: the pusher arrives at its target and coverage never moves again.

## Part 6: the LLM-written policy

Asked Gemini to write a PushT policy. It produced clean code that scores 0/100. Two measured bugs:

- It hardcoded goal (400, 400). The real goal is (256, 256, π/4) — it aimed at an empty corner.
- It treated the action as a **displacement** from the pusher's position. The action is an **absolute target pixel**; adding small steps to it teleports the pusher around the table.
- Lesson: AI output still needs a reviewer who can read `env.action_space` and measure. The model cannot run your environment.

## Part 7: The official Diffusion Policy

The published checkpoint (`lerobot/diffusion_pusht`, ~1 GB) is a DiffusionPolicy: it sees 2 image+state frames, runs a 16-step denoising loop, and emits 16 future actions, consuming 8 per step. About 38 model calls per episode.

- 0.4.4 loads weights through a *pre/post-processor* system (`policy_preprocessor.json` + normalization stats). 

```
hf download lerobot/diffusion_pusht README.md config.json train_config.json eval_info.json --local-dir weeks/week-03/model/diffusion_pusht

hf download lerobot/diffusion_pusht model.safetensors --local-dir weeks/week-03/model/diffusion_pusht
```

A success clip from that eval (the T slides to the goal):

<video src="weeks/week-03/outputs/dp_official_eval/videos/pusht_0/eval_episode_0.mp4" loop autoplay muted controls width="342" height="342" style="object-fit: cover;"></video><br>



In [ ]:
#Random Policy, 1000 Episodes,

import shutil, time
from pathlib import Path
import numpy as np
import gymnasium as gym
import gym_pusht
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.envs.utils import preprocess_observation

MIG  = "/Users/snudurupati/Projects/physical-ai-data-architect/weeks/week-03/model/diffusion_pusht_migrated"
ROOT = "/Users/snudurupati/Projects/physical-ai-data-architect/weeks/week-03/data/pusht_sim"
DEV  = "mps"
shutil.rmtree(ROOT, ignore_errors=True)          # fresh start

policy = DiffusionPolicy.from_pretrained(MIG)
policy.to(DEV); policy.eval()
pre, post = make_pre_post_processors(
    policy_cfg=policy.config, pretrained_path=MIG,
    preprocessor_overrides={"device_processor": {"device": DEV},
                            "rename_observations_processor": {"rename_map": {}}})

env = gym.make("gym_pusht/PushT-v0", obs_type="pixels_agent_pos", render_mode="rgb_array")

features = {
    "observation.image": {"dtype": "image",   "shape": (96, 96, 3), "names": ["height", "width", "channel"]},
    "observation.state": {"dtype": "float32", "shape": (2,), "names": ["x", "y"]},
    "action":            {"dtype": "float32", "shape": (2,), "names": ["x", "y"]},
    "is_success":        {"dtype": "bool",    "shape": (1,), "names": None},
}
ds = LeRobotDataset.create("pusht_sim", fps=10, features=features,
                           root=ROOT, robot_type="pusht", use_videos=False)

def rollout(make_action, n_episodes, tag):
    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        policy.reset()
        done, n = False, 0
        while not done and n < 300:
            action = make_action(obs, policy, pre, post, DEV)
            obs, r, term, trunc, info = env.step(action)
            ds.add_frame({
                "observation.image": obs["pixels"],
                "observation.state": np.asarray(obs["agent_pos"], dtype=np.float32),
                "action": np.asarray(action, dtype=np.float32),
                "is_success": np.array([bool(info["is_success"])], dtype=bool),
                "task": tag,
            })
            done = term or trunc
            n += 1
        ds.save_episode()
        if (ep + 1) % 50 == 0:
            print(f"{tag}: {ep+1}/{n_episodes} episodes done", flush=True)

t0 = time.time()
rollout(lambda obs, p, pre, post, dev: env.action_space.sample().astype(np.float32),
        1000, "random policy")
ds.finalize()
print(f"RANDOM 1000 done in {time.time()-t0:.0f}s -> {ROOT}")

In [ ]:
# The DIFFUSION Policy, 50 Episodes

N_DIFFUSION = 50          # ~100 s/episode → ~85 min; lower if time is tight

ds = LeRobotDataset("pusht_sim", root=ROOT)   # reopen the finalized dataset

def diffusion_action(obs, p, pre, post, dev):
    t = pre(preprocess_observation(obs))       # keys → policy format, batch+device+normalize
    with torch.inference_mode():
        a = p.select_action(t)                 # uses internal 8-action queue between calls
    a = post(a)                                # un-normalize with checkpoint stats
    return np.asarray(a.to("cpu"), dtype=np.float32).squeeze(0)

t0 = time.time()
rollout(diffusion_action, N_DIFFUSION, "diffusion")
ds.finalize()
print(f"DIFFUSION {N_DIFFUSION} done in {time.time()-t0:.0f}s")

Measured output of that cell:

```
random policy: 1000 episodes, 300000 frames,   0 success episodes
diffusion:       50 episodes,  10759 frames,  36 success episodes
```

Next for Parts 9–10: `01_profiler.ipynb` on the same dataset writes `results/pusht_sim.json`; its comparison cell puts it beside the two SO-101 profiles.